In [22]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

In [23]:
df = pd.read_csv('../Data/healthcare_data_encoded.csv')
df = df.set_index('patient_id')

In [24]:
cols_to_drop = ['age_group', 'high_glucose', 'bmi_category', 'lifestyle_risk', 'bmi_was_missing']
X = df.drop(columns=['stroke_event'] + cols_to_drop)
y = df['stroke_event']

In [25]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, stratify=y, random_state=42
)

In [26]:
neg, pos = y_train.value_counts()
spw = neg / pos

In [27]:
from sklearn.pipeline import Pipeline
from sklearn.model_selection import StratifiedKFold, RandomizedSearchCV
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import (
    precision_recall_curve,
    recall_score,
    precision_score,
    average_precision_score,
    roc_auc_score,
    confusion_matrix,
    f1_score,
    accuracy_score
)
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from xgboost import XGBClassifier

In [28]:
# Helper functions for threshold optimisation and evaluation

# F2 optimized threshold

def get_f2_threshold(y_true, y_probs):
    precision_arr, recall_arr, thresholds = precision_recall_curve(y_true, y_probs)
    f2 = (5 * precision_arr * recall_arr) / (4 * precision_arr + recall_arr + 1e-10)
    idx = np.argmax(f2)
    return thresholds[min(idx, len(thresholds) - 1)]

In [29]:
# evaluation metrics at a given threshold

def evaluate_model(name, y_true, y_probs, threshold, label):
    y_pred = (y_probs >= threshold).astype(int)
    tn, fp, fn, tp = confusion_matrix(y_true, y_pred).ravel()

    precision = precision_score(y_true, y_pred, zero_division=0)
    recall = recall_score(y_true, y_pred)

    return {
        'Model': name,
        'Threshold_Type': label,
        'Threshold': round(threshold, 4),
        'Accuracy': round(accuracy_score(y_true, y_pred), 4),
        'Precision': round(precision, 4),
        'Recall': round(recall, 4),
        'F1': round(f1_score(y_true, y_pred, zero_division=0), 4),
        'F2': round((5 * precision * recall) /
                    (4 * precision + recall + 1e-10), 4),
        'PR-AUC': round(average_precision_score(y_true, y_probs), 4),
        'ROC-AUC': round(roc_auc_score(y_true, y_probs), 4),
        'TP': int(tp),
        'TN': int(tn),
        'FP': int(fp),
        'FN': int(fn),
        'Flagged%': round((y_probs >= threshold).mean() * 100, 1)
    }
    

In [30]:
skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
results = []
pr_curves = {}

In [31]:
# Logistic pipeline

lr = Pipeline([
    ('scaler', StandardScaler()),
    ('lr', LogisticRegression(class_weight='balanced', max_iter=5000))
])

lr_params = {
    'lr__C': [0.001, 0.01, 0.1, 1, 10],
    'lr__l1_ratio': [1],
    'lr__solver': ['liblinear', 'saga']
}

lr_search = RandomizedSearchCV(
    lr, lr_params, n_iter=15,
    scoring='average_precision', cv=skf, n_jobs=-1, random_state=42
)
lr_search.fit(X_train, y_train)

lr_probs = lr_search.best_estimator_.predict_proba(X_test)[:, 1]
lr_thr = get_f2_threshold(y_test, lr_probs)

rec, prec, _ = precision_recall_curve(y_test, lr_probs)
pr_curves['Logistic Regression'] = (rec, prec)

results.append(evaluate_model('Logistic Regression', y_test, lr_probs, 0.5, 'Default (0.5)'))
results.append(evaluate_model('Logistic Regression', y_test, lr_probs, lr_thr, 'F2-Optimal'))

/opt/homebrew/Caskroom/miniconda/base/envs/dataxplore__env/lib/python3.11/site-packages/sklearn/model_selection/_search.py:324: UserWarning: The total space of parameters 10 is smaller than n_iter=15. Running 10 iterations. For exhaustive searches, use GridSearchCV.
  warnings.warn(
/opt/homebrew/Caskroom/miniconda/base/envs/dataxplore__env/lib/python3.11/site-packages/sklearn/linear_model/_sag.py:348: ConvergenceWarning: The max_iter was reached which means the coef_ did not converge
  warnings.warn(
/opt/homebrew/Caskroom/miniconda/base/envs/dataxplore__env/lib/python3.11/site-packages/sklearn/linear_model/_sag.py:348: ConvergenceWarning: The max_iter was reached which means the coef_ did not converge
  warnings.warn(
/opt/homebrew/Caskroom/miniconda/base/envs/dataxplore__env/lib/python3.11/site-packages/sklearn/linear_model/_sag.py:348: ConvergenceWarning: The max_iter was reached which means the coef_ did not converge
  warnings.warn(


In [32]:
# Random Forest pipeline

rf = Pipeline([
    ('scaler', StandardScaler()),
    ('rf', RandomForestClassifier(class_weight='balanced', n_jobs=-1))
])

rf_params = {
    'rf__n_estimators': [100, 200],
    'rf__max_depth': [3, 5, None],
    'rf__min_samples_leaf': [1, 5, 10]
}

rf_search = RandomizedSearchCV(
    rf, rf_params, n_iter=15,
    scoring='average_precision', cv=skf, n_jobs=-1, random_state=42
)
rf_search.fit(X_train, y_train)

rf_probs = rf_search.best_estimator_.predict_proba(X_test)[:, 1]
rf_thr = get_f2_threshold(y_test, rf_probs)

rec, prec, _ = precision_recall_curve(y_test, rf_probs)
pr_curves['Random Forest'] = (rec, prec)

results.append(evaluate_model('Random Forest', y_test, rf_probs, 0.5, 'Default (0.5)'))
results.append(evaluate_model('Random Forest', y_test, rf_probs, rf_thr, 'F2-Optimal'))

In [33]:
# XGBoost pipeline

xgb = Pipeline([
    ('scaler', StandardScaler()),
    ('xgb', XGBClassifier(scale_pos_weight=spw, eval_metric='logloss', verbosity=0))
])

xgb_params = {
    'xgb__n_estimators': [100, 200],
    'xgb__max_depth': [3, 4],
    'xgb__learning_rate': [0.01, 0.1]
}

xgb_search = RandomizedSearchCV(
    xgb, xgb_params, n_iter=8,
    scoring='average_precision', cv=skf, n_jobs=-1, random_state=42
)
xgb_search.fit(X_train, y_train)

xgb_probs = xgb_search.best_estimator_.predict_proba(X_test)[:, 1]
xgb_thr = get_f2_threshold(y_test, xgb_probs)

rec, prec, _ = precision_recall_curve(y_test, xgb_probs)
pr_curves['XGBoost'] = (rec, prec)

results.append(evaluate_model('XGBoost', y_test, xgb_probs, 0.5, 'Default (0.5)'))
results.append(evaluate_model('XGBoost', y_test, xgb_probs, xgb_thr, 'F2-Optimal'))

In [34]:
results_df = pd.DataFrame(results)

cols = [
    'Model', 'Threshold_Type', 'Threshold',
    'Accuracy', 'Precision', 'Recall', 'F1', 'F2',
    'PR-AUC', 'ROC-AUC',
    'TP', 'TN', 'FP', 'FN',
    'Flagged%'
]

print(results_df[cols].to_string(index=False))


              Model Threshold_Type  Threshold  Accuracy  Precision  Recall     F1     F2  PR-AUC  ROC-AUC  TP  TN  FP  FN  Flagged%
Logistic Regression  Default (0.5)     0.5000    0.7476     0.1414    0.82 0.2412 0.4184  0.2478   0.8429  41 723 249   9      28.4
Logistic Regression     F2-Optimal     0.6877    0.8474     0.2088    0.76 0.3276 0.4974  0.2478   0.8429  38 828 144  12      17.8
      Random Forest  Default (0.5)     0.5000    0.8611     0.2051    0.64 0.3107 0.4494  0.2675   0.8304  32 848 124  18      15.3
      Random Forest     F2-Optimal     0.5148    0.8738     0.2238    0.64 0.3316 0.4665  0.2675   0.8304  32 861 111  18      14.0
            XGBoost  Default (0.5)     0.5000    0.6820     0.1170    0.84 0.2054 0.3757  0.2450   0.8446  42 655 317   8      35.1
            XGBoost     F2-Optimal     0.6488    0.8317     0.1950    0.78 0.3120 0.4875  0.2450   0.8446  39 811 161  11      19.6
